In [7]:
import os
import csv
from bs4 import BeautifulSoup
from pathlib import Path
import pandas as pd

In [8]:
def extract_abstract(soup):
    """
    Extrae el abstract del HTML de PubMed Central.
    Maneja diferentes estructuras HTML comunes en PMC.
    """
    abstract_text = ""
    
    # Estrategia 1: Buscar por la clase 'abstract' dentro de una section
    abstract_section = soup.find('section', class_='abstract')
    
    if abstract_section:
        # Buscar el <h2> con el título "Abstract" y removerlo
        h2_title = abstract_section.find('h2')
        if h2_title:
            h2_title.decompose()
        
        # Obtener todos los párrafos <p> dentro de la sección abstract
        paragraphs = abstract_section.find_all('p')
        if paragraphs:
            abstract_text = ' '.join([p.get_text(strip=True) for p in paragraphs])
        else:
            # Si no hay párrafos, obtener todo el texto
            abstract_text = abstract_section.get_text(separator=' ', strip=True)
    
    # Estrategia 2: Buscar por div con clase o id 'abstract'
    if not abstract_text:
        abstract_div = soup.find('div', class_='abstract') or \
                      soup.find('div', {'id': 'abstract'})
        if abstract_div:
            for title in abstract_div.find_all(['h2', 'h3', 'title']):
                title.decompose()
            abstract_text = abstract_div.get_text(separator=' ', strip=True)
    
    # Estrategia 3: Buscar en meta tags
    if not abstract_text:
        meta_abstract = soup.find('meta', {'name': 'description'})
        if meta_abstract and meta_abstract.get('content'):
            abstract_text = meta_abstract.get('content')
    
    return abstract_text.strip()


def extract_conclusions(soup):
    """
    Extrae las conclusiones del HTML de PubMed Central.
    """
    conclusions_text = ""
    
    # Buscar secciones de conclusiones con diferentes nombres posibles
    possible_headers = ['conclusion', 'conclusions', 'concluding remarks', 
                       'summary', 'conclusions and perspectives']
    
    # Buscar en todos los headers (h2, h3, h4)
    for header in soup.find_all(['h2', 'h3', 'h4']):
        header_text = header.get_text().lower().strip()
        if any(term in header_text for term in possible_headers):
            # Obtener el siguiente elemento hermano
            next_elem = header.find_next_sibling()
            if next_elem:
                # Si es un párrafo, obtenerlo
                if next_elem.name == 'p':
                    conclusions_text = next_elem.get_text(separator=' ', strip=True)
                # Si es otra sección, obtener todos los párrafos dentro
                else:
                    paragraphs = []
                    current = next_elem
                    while current and current.name not in ['h2', 'h3', 'h4']:
                        if current.name == 'p':
                            paragraphs.append(current.get_text(strip=True))
                        current = current.find_next_sibling()
                    conclusions_text = ' '.join(paragraphs)
                break
    
    # También buscar por ID o clase
    if not conclusions_text:
        conclusion_div = soup.find('div', {'id': lambda x: x and 'conclusion' in x.lower()}) or \
                        soup.find('section', {'id': lambda x: x and 'conclusion' in x.lower()})
        if conclusion_div:
            conclusions_text = conclusion_div.get_text(separator=' ', strip=True)
    
    return conclusions_text.strip()


def process_pmc_htmls(folder_path='pmc_htmls', output_csv='pubmed_data.csv'):
    """
    Procesa todos los archivos HTML en la carpeta y crea el CSV.
    """
    folder = Path(folder_path)
    if not folder.exists():
        print(f"Error: La carpeta '{folder_path}' no existe")
        return
    
    results = []
    html_files = list(folder.glob('*.html'))
    
    if not html_files:
        print(f"No se encontraron archivos HTML en '{folder_path}'")
        return
    
    print(f"Procesando {len(html_files)} archivos...\n")
    
    for html_file in html_files:
        pmc_id = html_file.stem
        
        try:
            with open(html_file, 'r', encoding='utf-8') as f:
                html_content = f.read()
            
            soup = BeautifulSoup(html_content, 'html.parser')
            
            abstract = extract_abstract(soup)
            conclusions = extract_conclusions(soup)
            
            # Mostrar preview del abstract extraído
            preview = abstract[:100] + '...' if len(abstract) > 100 else abstract
            print(f"✓ {pmc_id}")
            print(f"  Abstract: {preview if abstract else '[No encontrado]'}")
            print(f"  Conclusions: {'[Encontrado]' if conclusions else '[No encontrado]'}")
            print()
            
            results.append({
                'pmc_id': pmc_id,
                'abstract': abstract,
                'conclusions': conclusions
            })
            
        except Exception as e:
            print(f"✗ Error procesando {pmc_id}: {str(e)}\n")
            results.append({
                'pmc_id': pmc_id,
                'abstract': '',
                'conclusions': ''
            })
    
    # Escribir resultados al CSV
    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['pmc_id', 'abstract', 'conclusions']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(results)
    
    print(f"{'='*60}")
    print(f"✓ Archivo CSV creado: {output_csv}")
    print(f"Total de artículos procesados: {len(results)}")
    
    # Estadísticas
    with_abstract = sum(1 for r in results if r['abstract'])
    with_conclusions = sum(1 for r in results if r['conclusions'])
    print(f"Artículos con abstract: {with_abstract}/{len(results)}")
    print(f"Artículos con conclusiones: {with_conclusions}/{len(results)}")


if __name__ == "__main__":
    # Ejecutar el procesamiento
    #process_pmc_htmls()
    
    # Si quieres personalizar las rutas:
    process_pmc_htmls(folder_path='pmc_htmls', output_csv='pubmed_data2.csv')

Procesando 572 archivos...

✓ PMC11999716
  Abstract: We investigate the bioremediation potential of the microbiome of the Gowanus Canal, a contaminated w...
  Conclusions: [Encontrado]

✓ PMC7076552
  Abstract: Recent advances in the routine access to space along with increasing opportunities to perform plant ...
  Conclusions: [Encontrado]

✓ PMC4469364
  Abstract: The oocytes of the African clawed frog (Xenopus laevis) comprise one of the most widely used membran...
  Conclusions: [No encontrado]

✓ PMC3251573
  Abstract: Extra-intestinal pathogenicE. coli(ExPEC), including avian pathogenicE. coli(APEC), pose a considera...
  Conclusions: [No encontrado]

✓ PMC10503492
  Abstract: Spaceflight poses risks to the central nervous system (CNS), and understanding neurological response...
  Conclusions: [No encontrado]

✓ PMC9953463
  Abstract: Efforts to understand the impact of spaceflight on the human body stem from growing interest in long...
  Conclusions: [No encontrado]

✓ PMC58266

In [13]:
df = pd.read_csv("pubmed_data2.csv")
df.head()

,pmc_id,abstract,conclusions
0,PMC11999716,We investigate the bioremediation potential of...,The microbiome of the Gowanus Canal is a biote...
1,PMC7076552,Recent advances in the routine access to space...,As the volume of spaceflight omics-level data ...
2,PMC4469364,The oocytes of the African clawed frog (Xenopu...,NaN
3,PMC3251573,"Extra-intestinal pathogenicE. coli(ExPEC), inc...",NaN
4,PMC10503492,Spaceflight poses risks to the central nervous...,NaN


In [15]:
dfr = pd.read_csv("articles_data_updated.csv")
dfr["Abstract"] = df["abstract"]
dfr.head()

,PMC_ID,Title,Authors,Introduction,Development/Methods,Results,Discussion,References_Count,References,Conclusions,Abstract
0,PMC4136787,Mice in Bion-M 1 Space Mission: Training and S...,Alexander Andreev-Andrievskiy; Anfisa Popova; ...,"After a 16-year hiatus, Russia resumed in 2013...",The study was approved by IACUC of MSU Institu...,Living conditions for animals considered optim...,Living conditions for animals considered optim...,37,2007 Animals in space Vestnik Rossijskoj Akade...,The microbiome of the Gowanus Canal is a biote...,We investigate the bioremediation potential of...
1,PMC3630201,Microgravity Induces Pelvic Bone Loss through ...,Elizabeth A. Blaber; Natalya Dvorochkin; Chial...,"On Earth, at 1 g, mechanical loading of mammal...",All experimental animal procedures for STS-131...,All flight and ground control mice were observ...,"In this study, we investigated cellular and mo...",74,2000 Historical overview of the Bion project J...,As the volume of spaceflight omics-level data ...,Recent advances in the routine access to space...
2,PMC11988870,Microgravity and Cellular Biology: Insights in...,Nelson Adolfo López Garzón; María Virginia Pin...,"Microgravity, a condition characterized by min...",A comprehensive literature review was conducte...,NaN,Recent research demonstrates that microgravity...,70,2003 Genetic models in applied physiology: sel...,NaN,The oocytes of the African clawed frog (Xenopu...
3,PMC7998608,Selective Proliferation of Highly Functional A...,Takanobu Mashiko; Koji Kanayama; Natsumi Saito...,Human adipose-derived stem cells (hASCs) are e...,Human lipoaspirates were obtained from 12 heal...,Cells were expanded for three passages before ...,"Through novel advances in cell biology, adult ...",48,2013 Effects of spaceflight and ground recover...,Plasmids Size (bp) Inc group GC% N° ORFs Start...,"Extra-intestinal pathogenicE. coli(ExPEC), inc..."
4,PMC5587110,Microgravity validation of a novel system for ...,Macarena Parra; Jimmy Jung; Travis D. Boone; L...,The ISS National Laboratory is a unique resear...,"In order to validate the system, a number of g...",In order to assess the functionality of PCR in...,One of the major obstacles to space exploratio...,38,2013 Changes in Mouse Thymus and Spleen after ...,Spaceflight poses risks to the central nervous...,Spaceflight poses risks to the central nervous...


In [19]:
# Save the updated DataFrame
dfr.to_csv("articles_data_updated.csv", index=False, encoding='utf-8')
dfr["Abstract"].head()
dfr["References"].head()

0    2007 Animals in space Vestnik Rossijskoj Akade...
1    2000 Historical overview of the Bion project J...
2    2003 Genetic models in applied physiology: sel...
3    2013 Effects of spaceflight and ground recover...
4    2013 Changes in Mouse Thymus and Spleen after ...
Name: References, dtype: object